# **Projeto de Aprendizagem: Análise de Dados do Censo Demográfico de 2022 (IBGE) com Pandas e Matplotlib**

### **Descrição das Base de Dados utilizadas**

#### Base de Dados 'censo' do ano de 2022

| Coluna | Descrição |
| :--- | :--- |
| **id_municipio** | ID Município |
| **sigla_uf** | Sigla UF |
| **domicilios** | Domicílios particulares permanentes ocupados |
| **populacao** | Moradores em domicílios particulares permanentes ocupados |
| **area** | Área do município (km²) |
| **taxa_alfabetizacao** | Taxa de alfabetização das pessoas de 15 anos ou mais de idade |
| **indice_envelhecimento** | Índice de envelhecimento |
| **idade_mediana** | Idade mediana |
| **razao_sexo** | Razão de sexo |
| **populacao_indigena** | Moradores indígenas |
| **populacao_indigena_terra_indigena** | Moradores indígenas em terras indígenas |
| **populacao_quilombola** | Moradores quilombolas |
| **populacao_quilombola_territorio_quilombola** | Moradores quilombolas em territórios quilombolas |

##### Link de acesso: <https://basedosdados.org/dataset/08a1546e-251f-4546-9fe0-b1e6ab2b203d?table=707fd42e-95e0-4856-922f-fcbb55db913a>

<br>

#### Base de Dados 'municipios' de 2024


| Coluna | Descrição |
| :--- | :--- |
| **id_uf** | ID do Estado |
| **nome_uf** | Nome do Estado |
| **id_municipio** | ID do Município |
| **nome_municipio** | nome do Município |

##### Link de acesso: <https://www.ibge.gov.br/geociencias/organizacao-do-territorio/estrutura-territorial/23701-divisao-territorial-brasileira.html?=&t=downloads&utm_source=landing&utm_medium=explica&utm_campaign=codmun>

*Observação: o arquivo original contém mais colunas, porém apenas essas 4 foram utilizadas neste projeto.*

### Importação das bibliotecas

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

### Importação das base de dados

In [ ]:
# O 'r' antes da string ajuda a ler caminhos do Windows com barras invertidas (\)

censo = pd.read_csv(r'..\dados\censo.csv')

municipios = pd.read_excel(r'..\dados\municipios.xlsx')

### Análise inicial dos dados

In [ ]:
# Exibe as 5 primeiras linhas
censo.head()

In [ ]:
# Exibe as 5 últimas linhas
censo.tail()

In [ ]:
# Exibe apenas os dados de uma coluna 'sigla_uf'
censo['sigla_uf']

### Manipulação e Filtros

In [ ]:
# Cria uma lista de colunas desejadas
cols = ['sigla_uf', 'populacao']

# Filtra o dataframe para exibir apenas as duas colunas ('sigla_uf' e 'populacao')
censo[cols]

In [ ]:
# Exemplo de concatenação de strings: Adiciona " - BR" ao final de cada sigla de UF
censo['sigla_uf'] + ' - BR'

In [ ]:
# Fitros avançados
# Filtro 1: Seleciona apenas linhas onde o estado é MS
filtro_uf = censo['sigla_uf'] == 'MS'

# Filtro 2: Seleciona municípios com população acima de 100.000 habitantes
filtro_populacao = censo['populacao'] > 100000

# Aplica os dois filtros com o operador AND
censo[filtro_uf & filtro_populacao]

In [ ]:
# Aplica os dois filtros com o operador OR
censo[filtro_uf | filtro_populacao]

In [ ]:
# Seleciona municípios com população abaixo de 100.000 habitantes
filtro_populacao = censo['populacao'] < 100000

# Aplica o filtro com o operador NOT, ou seja, ao negar, irá filtrar todas as populações com mais de 100.000 habitantes
censo[~filtro_populacao]

### Agregações Simples (Group By)

In [ ]:
# Agrupa os dados pela Sigla da UF e calcula estatísticas (soma e máximo)
censo.groupby('sigla_uf').agg(
    populacao_total = ('populacao', 'sum'), # Soma da população por estado
    domicilios_total = ('domicilios', 'sum'), # Soma dos domicílios por estado
    maior_populacao = ('populacao', max) # Maior população encontrada naquele estado
)

### Combinação de Dados (Merge)

In [ ]:
# Realiza um INNER JOIN entre a tabela do Censo e a tabela de Municípios pela coluna 'id_municipio'
merge = pd.merge(censo, municipios, how='inner', on='id_municipio')

# Visualiza as 5 primeiras linhas da junção
merge.head()

### Configuração do tipo de dado FLOAT

In [ ]:
pd.options.display.float_format = '{:,.2f}'.format

### Agregação Complexa

In [ ]:
# Define as colunas que serão usadas para o agrupamento
grupo = ['sigla_uf', 'nome_uf']

# as_index=False mantém as colunas agrupadoras como colunas normais, não como índice
dados_por_uf = merge.groupby(grupo, as_index=False).agg(
    domicilios_total = ('domicilios', 'sum'),
    populacao_total = ('populacao', 'sum'),
    populacao_indigena_total = ('populacao_indigena', 'sum'),
    populacao_indigena_terra_indigena = ('populacao_indigena_terra_indigena', 'sum'),
    populacao_quilombola_total = ('populacao_quilombola', 'sum'),
    populacao_quilombola_territorio_quilombola_total = ('populacao_quilombola_territorio_quilombola', 'sum')
)

### Ranking e Seleção

In [ ]:
# Define as colunas que serão utilizadas
cols_populacao_indigena_por_uf = ['sigla_uf', 'populacao_indigena_total']

# Ordena do maior para o menor a coluna 'populacao_indigena_total' e pega os 5 primeiros (Top 5)
top_5_populacao_indigena_por_uf = dados_por_uf[cols_populacao_indigena_por_uf].sort_values(
    by='populacao_indigena_total', ascending=False).head(5)

# Exibe o resultado
top_5_populacao_indigena_por_uf

### Visualização (Gráficos)

In [ ]:
# Gráfico de barras vertical
grafico_barra_vertical = plt.bar(
    top_5_populacao_indigena_por_uf['sigla_uf'], # Eixo X
    top_5_populacao_indigena_por_uf['populacao_indigena_total'], # Eixo Y
    color='skyblue' # Cor das barras
)
plt.xlabel('Estados') # Título do Eixo X
plt.ylabel('População') # Título do Eixo Y
plt.title('Top 5 maiores Estados do Brasil com populações Indígenas') # Título do Gráfico

plt.bar_label(grafico_barra_vertical, label_type='edge') # Rótulos: Adiciona os números acima das barras
plt.show()

In [ ]:
# Gráfico de barras horizontal
grafico_barra_horizontal = plt.barh(
    top_5_populacao_indigena_por_uf['sigla_uf'], # Eixo X
    top_5_populacao_indigena_por_uf['populacao_indigena_total'], # Eixo Y
    color='skyblue' # Cor das barras
)
plt.xlabel('Estados') # Título do Eixo X
plt.ylabel('População') # Título do Eixo Y
plt.title('Top 5 maiores Estados do Brasil com populações Indígenas') # Título do Gráfico

plt.bar_label(grafico_barra_horizontal, label_type='edge') # Rótulos: Adiciona os números ao lado das barras
plt.show()

### Exportação

In [ ]:
# Salva o dataframe em um arquivo CSV 
top_5_populacao_indigena_por_uf.to_csv(r'..\saida\top_5_populacao_indigena_por_uf.csv')